In [ ]:
from slicer.ScriptedLoadableModule import *
from __main__ import vtk, qt, ctk, slicer
from vtk.util import numpy_support
import SimpleITK as sitk
import sitkUtils as su
import os

def ct_registration(fixed_image, mobile_image, transformation):
    resampler = sitk.ResampleImageFilter()
    resampler.SetReferenceImage(fixed_image)
    resampler.SetInterpolator(sitk.sitkLinear)
    resampler.SetDefaultPixelValue(-1000)
    resampler.SetTransform(transformation)
    image_registration = resampler.Execute(mobile_image)
    return image_registration

def rtdose_registration(fixed_image, mobile_image, transformation):
    resampler = sitk.ResampleImageFilter()
    resampler.SetReferenceImage(fixed_image)
    resampler.SetInterpolator(sitk.sitkLinear)
    resampler.SetDefaultPixelValue(0)
    resampler.SetTransform(transformation)
    image_registration = resampler.Execute(mobile_image)
    return image_registration

def multires_registrations(fixed_image, moving_image, initial_transform, ImageSamplingPercentage):
    registration_method = sitk.ImageRegistrationMethod()
    registration_method.SetMetricAsMattesMutualInformation(numberOfHistogramBins = 50)
    registration_method.SetMetricSamplingStrategy(registration_method.RANDOM)
    registration_method.SetMetricSamplingPercentage(float(ImageSamplingPercentage)/100)
    registration_method.SetInterpolator(sitk.sitkLinear)
    registration_method.SetOptimizerAsGradientDescent(learningRate = 1.0, estimateLearningRate = registration_method.EachIteration, numberOfIterations = 100)
    registration_method.SetOptimizerScalesFromPhysicalShift()
    registration_method.SetInitialTransform(initial_transform)
    registration_method.SetShrinkFactorsPerLevel(shrinkFactors = [4, 2, 1])
    registration_method.SetSmoothingSigmasPerLevel(smoothingSigmas = [3, 2, 1])
    registration_method.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()
    final_transform = registration_method.Execute(fixed_image, moving_image)
    print('Final metric value: {0}'.format(registration_method.GetMetricValue()))
    print('Optimizer\'s stopping condition, {0}'.format(registration_method.GetOptimizerStopConditionDescription()))
    return final_transform

def main_image_Traitement_CT(fixed_mri, moving_ct, rtdose, percentage, filenameCT):
    moving_ct  = sitk.Cast(moving_ct, sitk.sitkFloat32)
    fixed_mri = sitk.Cast(fixed_mri, sitk.sitkFloat32)
    rtdose = sitk.Cast(rtdose, sitk.sitkFloat32)

    initial_transform = sitk.CenteredTransformInitializer(fixed_mri, moving_ct, sitk.Euler3DTransform(), sitk.CenteredTransformInitializerFilter.MOMENTS)

    medium_transform = multires_registrations(fixed_mri, moving_ct, initial_transform, percentage)

    moving_ct = ct_registration(fixed_mri, moving_ct, medium_transform)
    rtdose = rtdose_registration(fixed_mri, rtdose, medium_transform)

    su.PushVolumeToSlicer(moving_ct, name = 'XXX', className = 'vtkMRMLScalarVolumeNode')
    sitk.WriteTransform(medium_transform, "XXX" + filenameCT + ".h5")

    return moving_ct, rtdose

def main(input_data_directory, output_image_path):

    percentage = 20         #20 pour validation
    ct = 'CT'
    rtdose = 'RTDOSE'


    for filename in os.listdir(input_data_directory):
        if 'preMRI' in filename:
            input_image_path   = os.path.join(input_data_directory, filename)
            output_image_path  = os.path.join(output_image_path, filename)
            image = sitk.ReadImage(input_image_path)
            sitk.WriteImage(image, output_image_path)


    PatientID = None
    for filename_pre in os.listdir(input_data_directory):
        PatientID = filename_pre.split('_')[0]
            
        filename_CT   = PatientID + '_' + ct
        filename_dose = PatientID + '_' + rtdose

        try :
            if filename_CT + '.nii' in os.listdir(input_data_directory):

                input_data_directory = "XXX"
                output_image_path = "XXX"

                image_path = os.path.join(output_image_path, filename_pre)
                ct_path = os.path.join(input_data_directory, filename_CT + '.nii')
                rtdose_path = os.path.join(input_data_directory, filename_dose + '.nii')

                ct_image, rtdose_image = main_image_Traitement_CT(sitk.ReadImage(image_path), sitk.ReadImage(ct_path), sitk.ReadImage(rtdose_path), percentage, filename_CT)
                sitk.WriteImage(ct_image, output_image_path + '/' + filename_CT + '_w.nii')
                sitk.WriteImage(rtdose_image, output_image_path + '/' + filename_dose + '_w.nii')
                print("Finish")
                
        except :
            print('Error')
    return 0

input_data_directory   = "XXX"
output_image_path = "XXX"

main(input_data_directory, output_image_path)